# NLP Sentiment Analysis con Long-Short Term Memory (LSTM)

En este notebook implementaremos un clasificador de sentimientos en inglés utilizando la arquitectura de red LSTM. Utilizaremos un dataset local de sentimientos para entrenar y evaluar el modelo.

#### Referencias
- Dataset: sentimentdataset.csv
- [Long Short-Term Memory](https://www.researchgate.net/publication/13853244_Long_Short-Term_Memory#fullTextFileContent)

In [1]:
import importlib.metadata
installed_packages = [
    dist.metadata['Name'].lower() 
    for dist in importlib.metadata.distributions() 
    if dist.metadata['Name'] is not None
]
IN_COLAB = 'google-colab' in installed_packages

In [2]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets

"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


### Cargando el dataset
Cargaremos el archivo `sentimentdataset.csv` que contiene textos en inglés y sus respectivos sentimientos.

In [3]:
import pandas as pd
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
df_raw = pd.read_csv('sentimentdataset.csv')
df_raw['Sentiment'] = df_raw['Sentiment'].str.strip()
df_raw = df_raw[['Text', 'Sentiment']]
dataset = df_raw.to_dict('records')
print(f"Total de registros: {len(dataset)}")
df_raw.head()

Total de registros: 732


,Text,Sentiment
0,Enjoying a beautiful day at the park! ...,Positive
1,Traffic was terrible this morning. ...,Negative
2,Just finished an amazing workout! 💪 ...,Positive
3,Excited about the upcoming weekend getaway! ...,Positive
4,Trying out a new recipe for dinner tonight. ...,Neutral


Observemos uno de sus registros

In [4]:
#Observemos los primeros registros
dataset[0]

{'Text': ' Enjoying a beautiful day at the park!              ',
 'Sentiment': 'Positive'}

In [5]:
text_lengths = [len(row['Text']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

Texto más corto: 50
Texto más largo: 157
Longitud promedio: 87.59562841530055


In [6]:
import re
from collections import Counter

def simple_tokenizer(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z]+", " ", text)
    return text.strip().split()

# Construimos el vocabulario a partir de conjunto de datos.
token_counts = Counter()
for row in dataset:
    token_counts.update(simple_tokenizer(row["Text"]))

# 10k-2 porque el dataset es pequeño
top_n_tokens = list(token_counts.keys())[:10000-2]
vocab = {"[PAD]": 0, "[UNK]": 1}
for token in top_n_tokens:
    vocab[token] = len(vocab)

def tokenize_text(text, max_length=50):
    tokens = simple_tokenizer(text)
    ids = [vocab.get(tok, vocab["[UNK]"]) for tok in tokens[:max_length]]
    ids += [vocab["[PAD]"]] * (max_length - len(ids))
    return ids

In [7]:
print(f"Vocabulario: {len(vocab)} tokens")
print("Primeros 15 tokens:")
print(f"{top_n_tokens[:15]}")
print("15 tokens de en medio:")
print(f"{top_n_tokens[1000:1015]}")
print("Últimos 15 tokens:")
print(f"{top_n_tokens[-15:]}")

Vocabulario: 2532 tokens
Primeros 15 tokens:
['enjoying', 'a', 'beautiful', 'day', 'at', 'the', 'park', 'traffic', 'was', 'terrible', 'this', 'morning', 'just', 'finished', 'an']
15 tokens de en medio:
['seem', 'forever', 'reach', 'dismissive', 'glances', 'built', 'pleas', 'recognition', 'lie', 'floor', 'glass', 'disappointment', 'crafted', 'hands', 'only']
Últimos 15 tokens:
['pal', 'country', 'globe', 'enthusiasts', 'collaboration', 'spreading', 'track', 'regional', 'triumphs', 'company', 'fundraising', 'initiative', 'giving', 'multicultural', 'bringing']


Exploremos el vocabulario obtenido.

In [8]:
tokenized = tokenize_text("hola mundo", max_length=8)
tokenized

[1, 1, 0, 0, 0, 0, 0, 0]

Lo que obtenemos de vuelta son los ids de cada token según el vocabulario. Ahora algo importante que notamos aquí es el *padding*, durante el entrenamiento, queremos que las secuencias sean de tamaño fijo, para asi operar comodamente con matrices. Pero ya vimos que no todos los textos tienen la misma longitud. Entonces que hacer? para los que son más largos que una longitud dada simplemente cortamos, pero para los que son más cortos, debemos *rellenar* lo faltante con un *token especial de relleno o padding*. Y es justo lo que definimos allí, cuando la cadena es inferior a 8 **tokens**, entonces debemos hacer padding hasta que se cumplan los 8.

Si queremos saber a que token exactamente hacen referencia estos ids, simplemente revisamos el vocabulario que hemos construido:

In [9]:
id_2_token = {v: k for k, v in vocab.items()}
[id_2_token[token] for token in tokenized]

['[UNK]', '[UNK]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

Claramente vemos los 3 tokens como cadenas independientes (el padding se considera un token independiente).

### Definiendo el dataset de pytorch
Ahora podemos proceder a definir el dataset. Esto debería ser muy sencillo dado que nuestro dataset es pequeño y ya tenemos el tokenizador listo.

In [10]:
import torch
import numpy as np
from typing import Tuple, Dict
from torch.utils.data import Dataset
class SentimentDataset(Dataset):
    def __init__(self, tokenizer, data_list, seq_length: int = 128):
        self.tokenizer = tokenizer
        self.data = data_list
        self.seq_length = seq_length
        # Extraemos las categorías únicas de sentimientos
        self.id_2_class_map = dict(enumerate(sorted(list(set([row['Sentiment'] for row in data_list])))))
        self.class_2_id_map = {v: k for k, v in self.id_2_class_map.items()}
        self.num_classes = len(self.id_2_class_map)
    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        text, y = self.data[index]['Text'], self.data[index]['Sentiment']
        y = self.class_2_id_map[y]
        data = {'input_ids': torch.tensor(self.tokenizer(text, max_length=self.seq_length))}
        data['y'] = torch.tensor(y)
        return data
    def __len__(self):
        return len(self.data)

Ahora instanciaremos el dataset entero. Para este experimento, definiremos un tamaño máximo de secuencia de 2048 **tokens**. Que según nuestra intuición arriba, debería ser suficiente para la tarea.

Y luego, procedemos a hacer el train-val-test split y crear los dataloaders.

In [11]:
max_len = 64 
sentiment_dataset = SentimentDataset(tokenize_text, dataset, seq_length=max_len)
assert len(sentiment_dataset) == len(dataset)

## Definición del modelo LSTM

Ahora vamos a configurar un módulo pytorch simple para este problema. Vamos ha utilizar los embeddings, que vendrían siendo los vectores de palabra. Pytorch nos ofrece una capa con la que directamente podemos entrenarlos a partir de los token ids obtenidos. El resto consistirá en invocar una capa LSTM seguida de una capa densa para la clasificación.

Recordemos que las redes recurrentes como las LSTM por diseño enlazan todas las dimensiones del vector de entrada, formando así la secuencia, la estructura natural que necesitamos representar.

In [12]:
from torch.utils.data import random_split
from torch.utils.data import DataLoader

batch_size = 8
train_dataset, val_dataset, test_dataset = random_split(sentiment_dataset, lengths=[0.8, 0.1, 0.1])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

### Definición del clasificador

Finalmente, definimos el modelo en si. Este modelo constará de 3 capas:

- La tokenización, tal como la definimos anteriormente.
- El bloque LSTM, que acabamos de decinir.
- Una capa densa adicional que servirá como clasificador de aquello que nos entregue la capa del transformer.

Como este es un LightningModule, aquí definiremos el resto de funciones utilitarias para el entrenamiento de la tarea.

In [13]:
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import LightningModule, Trainer
from torchmetrics import Accuracy
class LSTMBlock(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
    
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, _) = self.lstm(embedded)
        return hidden[-1]
class SentimentClassifierWithLSTM(LightningModule):
    def __init__(self, vocab_size: int, num_classes: int, emb_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.num_classes = num_classes
        self.lstm = LSTMBlock(vocab_size, emb_dim, hidden_dim, num_classes)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
            nn.LogSoftmax(dim=1)
        )
        self.train_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.test_acc = Accuracy(task='multiclass', num_classes=num_classes)
    def forward(self, x):
        out = self.lstm(x)
        return self.classifier(out)
    def training_step(self, batch, batch_idx):
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        self.log('train-loss', loss, prog_bar=True)
        return loss
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-3)
model = SentimentClassifierWithLSTM(
    vocab_size=len(vocab) + 1, 
    num_classes=sentiment_dataset.num_classes, 
    emb_dim=128
)

Observemos el proceso de entrenamiento

In [14]:
%load_ext tensorboard

In [15]:
%tensorboard --logdir tb_logs/

Reusing TensorBoard on port 6006 (pid 17652), started 0:11:39 ago. (Use '!kill 17652' to kill it.)

Y como es de esperarse, realizaremos la validación contra el conjunto de prueba.

In [16]:
model.eval()
trainer.test(model, test_loader)

NameError: name 'trainer' is not defined

### Haciendo predicciones

Finalmente, vamos a hacer uso del modelo y ver que tan bueno es para la clasificación de noticias.

In [ ]:
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
predictions = torch.argmax(predictions, dim=-1)
predictions = [sentiment_dataset.id_2_class_map[pred] for pred in predictions.numpy()]

In [ ]:
import pandas as pd

test_indices = test_dataset.indices
df_eval = pd.DataFrame(data={
    "texto": [dataset[i]['Text'] for i in test_indices],
    "tokens": [tokenize_text(dataset[i]['Text']) for i in test_indices],
    "sentimiento": [dataset[i]['Sentiment'] for i in test_indices],
    'predicción': [sentiment_dataset.id_2_class_map[p] for p in torch.argmax(torch.cat(trainer.predict(model, test_loader)), dim=-1).numpy()]
}, index=test_indices)

df_eval['tokens_string'] = df_eval.tokens.apply(lambda t: ' '.join([id_2_token.get(i, '[UNK]') for i in t]))
df_eval = df_eval[["texto", "tokens", "tokens_string", "sentimiento", "predicción"]]
df_eval.head(15)

In [ ]:
errors = df_eval[df_eval['sentimiento'] != df_eval['predicción']]
errors.head(15)

## Conclusiones

- En este caso tenemos una tarea de clasificación de texto de múltiples clases.
- Estamos usando un bloque LSTM como featurizer, es decir lo usamos para extraer features de las secuencias de entrada con las cuales harémos predicciones luego.
- Nótese que de las capas LSTM, solo nos interesa la última, ya que esta recupera todas las operaciones enalazadas anteriores.
- Observamos que el modelo toma su tiempo en entrenar, esto es natural debido al diseño de las LSTM, donde por cada paso de tiempo se debe computar un gradiente, por lo que el computo es mucho mayor.